In [233]:
import pandas as pd
import psycopg2
import csv
from datetime import datetime

DATA EXTRACTION


In [234]:
customers_df = pd.read_csv(r'dataset\Rawdata\customers.csv')
products_df = pd.read_csv(r'dataset\Rawdata\products.csv')
regions_df = pd.read_csv(r'dataset\Rawdata\stores.csv')
calender_df = pd.read_csv(r'dataset\Rawdata\calender.csv')
sales_df = pd.read_csv(r'dataset\Rawdata\sales.csv')


DATA CLEANING AND TRANSFORMATION

In [ ]:
#checking the dataframes information
customers_df.info()
products_df.info()
regions_df.info()
calender_df.info()
sales_df.info()

In [ ]:
#cleaning and transforming customers dataframe
customers_df.dropna()
customers_df['join_date']= pd.to_datetime(customers_df['join_date'], format = 'mixed', errors = 'coerce').dt.tz_localize(None)
customers_df.copy().drop_duplicates().reset_index(drop=True)

In [ ]:
#cleaning and transforming products dataframe
products_df.dropna()
products_df.copy().drop_duplicates().reset_index(drop=True)

In [ ]:
#cleaning and transforming regions dataframe
regions_df.dropna()
regions_df.copy().drop_duplicates().reset_index(drop=True)

In [ ]:
#cleaning and transforming calendar dataframe
calender_df.dropna()
calender_df['date'] = pd.to_datetime(calender_df['date'], format = 'mixed', errors = 'coerce').dt.tz_localize(None)
calender_df.copy().drop_duplicates().reset_index(drop=True)

In [ ]:
#cleaning and transforming sales dataframe
sales_df.dropna()
sales_df['order_date']= pd.to_datetime(sales_df['order_date'], format = 'mixed', errors = 'coerce').dt.tz_localize(None)
sales_df.copy().drop_duplicates().reset_index(drop=True)

MERGING THE TABLES TO FORM THE SALES_FACT TABLE

In [240]:
sales_df = pd.merge(sales_df,products_df, on='product_id', how='left')

In [241]:
sales_df = pd.merge(sales_df,customers_df, on='customer_id', how='left')

In [242]:
sales_fact_df = pd.merge(sales_df,regions_df, on='store_id', how='left')

In [295]:
sales_fact_df = sales_fact_df[~sales_fact_df['product_id'].isin (['P0000','P0201'])]

In [296]:
#saving dataframes to Cleaneddata folder
customers_df.to_csv(r'dataset\Cleaneddata\customers.csv', index=False)
products_df.to_csv(r'dataset\Cleaneddata\products.csv', index=False)
regions_df.to_csv(r'dataset\Cleaneddata\regions.csv', index=False)
calender_df.to_csv(r'dataset\Cleaneddata\calender.csv', index=False)
sales_fact_df.to_csv(r'dataset\Cleaneddata\sales_fact.csv', index=False)


#Create derived columns like revenue and time features

In [ ]:
list(sales_fact_df.groupby('revenue')['order_date'])

CONNECT TO POSTGRES SERVER

In [257]:
#CODE TO CONNECT TO POSTGRES SERVER
def connect_to_postgres():
    connection = psycopg2.connect(
        host = 'localhost',
        port = 5432,
        database = 'chocolate_sales',
        user = 'postgres',
        password = 'Antigen.100$'
    ) 
    return connection

In [258]:
conn = connect_to_postgres()

LOADING RAW DATA INTO POSTGRES

In [264]:
#creating tables
def create_rawtables_postgres():
    conn = connect_to_postgres()
    cursor = conn.cursor()
    create_rawtables = '''CREATE SCHEMA IF NOT EXISTS operations;

                    DROP TABLE IF EXISTS operations.customers CASCADE;
                    DROP TABLE IF EXISTS operations.products CASCADE;
                    DROP TABLE IF EXISTS operations.stores CASCADE;
                    DROP TABLE IF EXISTS operation.calender CASCADE;
                    DROP TABLE IF EXISTS operations.sales CASCADE;
                    
                    CREATE TABLE IF NOT EXISTS operations.customers(
                    customer_id TEXT,
                    age INTEGER, 
                    gender TEXT,
                    loyalty_member INTEGER,
                    join_date DATE
                    );

                    CREATE TABLE IF NOT EXISTS operations.products(
                    product_id TEXT,
                    product_name TEXT, 
                    brand TEXT,
                    category TEXT,
                    cocoa_percent VARCHAR(500),
                    weight_g VARCHAR(500)
                    );

                    CREATE TABLE IF NOT EXISTS operations.stores(
                    store_id TEXT PRIMARY KEY,
                    store_name TEXT, 
                    city TEXT,
                    country TEXT,
                    store_type TEXT
                    );

                    CREATE TABLE IF NOT EXISTS operations.calendar(
                    date DATE,
                    year INTEGER,
                    month INTEGER,
                    day INTEGER,
                    week INTEGER,
                    day_of_week INTEGER
                    );

                    CREATE TABLE IF NOT EXISTS operations.sales(
                    order_id TEXT,
                    order_date DATE,
                    product_id TEXT,
                    store_id TEXT,
                    customer_id TEXT,
                    quantity INTEGER,
                    unit_price DECIMAL,
                    discount DECIMAL,
                    revenue DECIMAL,
                    cost DECIMAL,
                    profit DECIMAL
                    );'''
    cursor.execute(create_rawtables)
    conn.commit()
    cursor.close()
    conn.close()

In [265]:
create_rawtables_postgres()

In [267]:
#loading raw customers table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute('''
                            INSERT INTO operations.customers(customer_id, age, gender, loyalty_member,join_date)
                            VALUES(%s, %s, %s, %s, %s);''',
                            row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Rawdata\customers.csv'
load_data_into_postgres(csv_file_path)

In [ ]:
#load raw products table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
                cursor.execute('''
                           INSERT INTO operations.products(product_id, product_name, brand, category, cocoa_percent, weight_g)
                           VALUES(%s, %s, %s, %s, %s, %s);''',
                           row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Rawdata\products.csv'
load_data_into_postgres(csv_file_path)

In [269]:
#load raw stores table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute('''
                           INSERT INTO operations.stores(store_id, store_name, city, country, store_type)
                           VALUES(%s, %s, %s, %s, %s);''',
                           row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Rawdata\stores.csv'
load_data_into_postgres(csv_file_path)

In [270]:
#load raw calender table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute('''
                           INSERT INTO operations.calendar(date,year,month,day,week,day_of_week)
                           VALUES(%s, %s, %s, %s, %s, %s);''',
                           row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Rawdata\calender.csv'
load_data_into_postgres(csv_file_path)

In [271]:
#load sales table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
                cursor.execute('''
                           INSERT INTO operations.sales(order_id, order_date, product_id, store_id, customer_id,quantity,
                            unit_price, discount, revenue, cost, profit)
                           VALUES(%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);''',
                           row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Rawdata\sales.csv'
load_data_into_postgres(csv_file_path)

LOADING CLEANED DATA INTO POSTGRES

In [285]:
#creating tables
def create_tables_postgres():
    conn = connect_to_postgres()
    cursor = conn.cursor()
    create_tables = '''CREATE SCHEMA IF NOT EXISTS analytics;

                    DROP TABLE IF EXISTS analytics.customers CASCADE;
                    DROP TABLE IF EXISTS analytics.products CASCADE;
                    DROP TABLE IF EXISTS analytics.regions CASCADE;
                    DROP TABLE IF EXISTS analytics.sales_fact CASCADE;
                    
                    CREATE TABLE IF NOT EXISTS analytics.customers(
                    customer_id TEXT PRIMARY KEY,
                    age INTEGER, 
                    gender TEXT,
                    loyalty_member INTEGER,
                    join_date DATE
                    );

                    CREATE TABLE IF NOT EXISTS analytics.products(
                    product_id TEXT PRIMARY KEY,
                    product_name TEXT, 
                    brand TEXT,
                    category TEXT,
                    cocoa_percent VARCHAR(500),
                    weight_g VARCHAR(500)
                    );

                    CREATE TABLE IF NOT EXISTS analytics.regions(
                    store_id TEXT PRIMARY KEY,
                    store_name TEXT, 
                    city TEXT,
                    country TEXT,
                    store_type TEXT
                    );

                    CREATE TABLE IF NOT EXISTS analytics.sales_fact(
                    order_id TEXT PRIMARY KEY,
                    order_date DATE,
                    product_id TEXT,
                    store_id TEXT,
                    customer_id TEXT,
                    quantity INTEGER,
                    unit_price DECIMAL,
                    discount DECIMAL,
                    revenue DECIMAL,
                    cost DECIMAL,
                    profit DECIMAL,
                    product_name TEXT,
                    brand TEXT,
                    category TEXT,
                    cocoa_percent VARCHAR(500),
                    weight_g VARCHAR(500),
                    age INTEGER,
                    gender TEXT,
                    loyalty_member INTEGER,
                    join_date DATE,
                    store_name TEXT,
                    city TEXT,
                    country TEXT,
                    store_type TEXT,
                    FOREIGN KEY(customer_id) REFERENCES analytics.customers(customer_id),
                    FOREIGN KEY(product_id) REFERENCES analytics.products(product_id),
                    FOREIGN KEY(store_id) REFERENCES analytics.regions(store_id)                   
                    );'''
    cursor.execute(create_tables)
    conn.commit()
    cursor.close()
    conn.close()

In [286]:
create_tables_postgres()

In [287]:
#loading customers table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute('''
                            INSERT INTO analytics.customers(customer_id, age, gender, loyalty_member,join_date)
                            VALUES(%s, %s, %s, %s, %s);''',
                            row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Cleaneddata\customers.csv'
load_data_into_postgres(csv_file_path)



In [288]:
#load products table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
                cursor.execute('''
                           INSERT INTO analytics.products(product_id, product_name, brand, category, cocoa_percent, weight_g)
                           VALUES(%s, %s, %s, %s, %s, %s);''',
                           row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Cleaneddata\products.csv'
load_data_into_postgres(csv_file_path)

In [289]:
#load regions table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute('''
                           INSERT INTO analytics.regions(store_id, store_name, city, country, store_type)
                           VALUES(%s, %s, %s, %s, %s);''',
                           row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Cleaneddata\regions.csv'
load_data_into_postgres(csv_file_path)

In [297]:
#load sales table file into postgres
def load_data_into_postgres(csv_path):
    conn = connect_to_postgres()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
                cursor.execute('''
                           INSERT INTO analytics.sales_fact(order_id, order_date, product_id, store_id, customer_id,quantity,
                            unit_price, discount, revenue, cost, profit, product_name, brand, category, cocoa_percent, 
                            weight_g, age, gender, loyalty_member, join_date, store_name, city, country, store_type)
                           VALUES(%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, 
                            %s, %s, %s, %s,%s, %s);''',
                           row
                        )
    conn.commit()
    cursor.close()
    conn.close()
csv_file_path = r'dataset\Cleaneddata\sales_fact.csv'
load_data_into_postgres(csv_file_path)